# Taller 02 — Actor-Critic con componente de descubrimiento (DAC) en Asteroids

**Entorno:** `ALE/Asteroids-v5` (Gymnasium + ALE), 14 acciones discretas, observación 84×84×4 (grises, 4 frames apilados).

**Idea:** al actor-critic clásico se le agregan dos conjuntos de redes:

1. **Ensamble de críticos** $V^E_1..V^E_K$: su desacuerdo $\sigma_V(s)$ mide cuánto se puede confiar en el crítico.
   - El gradiente del actor se pondera con $w(s)=1/(1+\kappa\,\tilde\sigma_V(s))$ → el crítico no "engaña" al actor donde no sabe.
2. **Módulo de descubrimiento**: ensamble de $M$ modelos de dinámica $g_m(\psi(s),a)\to\psi(s')$ sobre un codificador aleatorio fijo $\psi$.
   - **Recompensa intrínseca:** $r^I_t = \frac{1}{d}\sum_j \mathrm{Var}_m[g_m(\psi(s_t),a_t)]_j$ (novedad epistémica), con su propio crítico $V^I$.
   - **"Preguntar por otra acción":** cuando $\sigma_V(s_t)$ supera su media móvil (o con prob. $\epsilon$), el agente consulta el mapa de novedad $U(s,\cdot)$ de **todas** las acciones y actúa con
     $\mu(a|s)\propto \pi_\theta(a|s)\,\exp(\eta_t\,\hat U(s,a))$.
   - Como se actúa con $\mu\neq\pi$, se corrige con $\bar\rho_t=\min(1,\pi_\theta(a_t|s_t)/b(a_t|s_t))$.

**Pérdidas**

$$\mathcal L_\pi=-\mathbb E\big[\bar\rho_t\,w(s_t)\,\hat A_t\log\pi_\theta(a_t|s_t)\big]-\beta\,\mathcal H(\pi_\theta),\qquad \hat A_t=A^E_t+c_I A^I_t$$

$$\mathcal L_V=\sum_k \mathbb E\big[m_{k,t}(V^E_k(s_t)-R^E_t)^2\big]+\mathbb E\big[(V^I(s_t)-R^I_t)^2\big],\qquad \mathcal L_D=\sum_m\mathbb E\big[m_{m,t}\|g_m(\psi(s_t),a_t)-\psi(s_{t+1})\|^2\big]$$

$A^E, A^I$ con GAE($\lambda$) (la intrínseca es no episódica); $m$ son máscaras bootstrap Bernoulli que mantienen diversos los ensambles.

> Entorno de ejecución recomendado: **GPU** (Entorno de ejecución → Cambiar tipo → T4).

## 0. Instalación

In [1]:
!pip install -q "gymnasium[atari]>=1.1" "ale-py>=0.10" imageio imageio-ffmpeg

## 1. Imports y entorno

In [2]:
# ============================================================
#  DAC: Discovery Actor-Critic  (Asteroids - Gymnasium / ALE)
# ============================================================
import os, time, json, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
import ale_py

gym.register_envs(ale_py)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Dispositivo:', DEVICE)

# 1. Entorno
# ------------------------------------------------------------
def make_env(env_id="ALE/Asteroids-v5", seed=0, render_mode=None):
    def thunk():
        # frameskip=1 en el entorno base: AtariPreprocessing hace el frame-skip (4)
        env = gym.make(env_id, frameskip=1, repeat_action_probability=0.25,
                       render_mode=render_mode)
        env = gym.wrappers.AtariPreprocessing(
            env, noop_max=30, frame_skip=4, screen_size=84,
            grayscale_obs=True, scale_obs=False, terminal_on_life_loss=False)
        env = gym.wrappers.FrameStackObservation(env, stack_size=4)
        env.reset(seed=seed)
        env.action_space.seed(seed)
        return env
    return thunk


def make_vec_env(env_id, num_envs, seed, asynchronous=True):
    fns = [make_env(env_id, seed + i) for i in range(num_envs)]
    kw = dict(autoreset_mode=gym.vector.AutoresetMode.SAME_STEP)
    venv = gym.vector.AsyncVectorEnv(fns, **kw) if asynchronous else gym.vector.SyncVectorEnv(fns, **kw)
    venv = gym.wrappers.vector.RecordEpisodeStatistics(venv)
    return venv

Dispositivo: cpu


## 2. Utilidades

In [3]:
# # 2. Utilidades
# ------------------------------------------------------------
class RunningMeanStd:
    """Media/varianza móviles (algoritmo paralelo de Chan)."""
    def __init__(self, shape=()):
        self.mean = np.zeros(shape, np.float64)
        self.var = np.ones(shape, np.float64)
        self.count = 1e-4

    def update(self, x):
        x = np.asarray(x, np.float64)
        b_mean, b_var, b_count = x.mean(0), x.var(0), x.shape[0]
        delta = b_mean - self.mean
        tot = self.count + b_count
        self.mean = self.mean + delta * b_count / tot
        m2 = self.var * self.count + b_var * b_count + delta ** 2 * self.count * b_count / tot
        self.var = m2 / tot
        self.count = tot


def nature_cnn(in_ch=4, out=512):
    return nn.Sequential(
        nn.Conv2d(in_ch, 32, 8, 4), nn.ReLU(),
        nn.Conv2d(32, 64, 4, 2), nn.ReLU(),
        nn.Conv2d(64, 64, 3, 1), nn.ReLU(),
        nn.Flatten(), nn.Linear(64 * 7 * 7, out), nn.ReLU())


def ortho_init(m, gain=math.sqrt(2)):
    if isinstance(m, (nn.Linear, nn.Conv2d)):
        nn.init.orthogonal_(m.weight, gain)
        nn.init.zeros_(m.bias)